# Feature Engineering — Rationale & Validation

The engineered features actually used in production live in
`src/feature_engineering.py` (`add_features`). This notebook does **not**
redefine that logic — it imports the real function, applies it to the raw
data, and visually validates that each engineered feature shows a
meaningful relationship with churn, which is the justification for
including it in the model.

This notebook is documentation / validation only. The executable pipeline
is `02_preprocessing.ipynb` → `04_model_training.ipynb` → `05_evaluation.ipynb`.

## 1. Imports & Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import add_features

plt.style.use("ggplot")
sns.set_theme(style="whitegrid", font_scale=1.1)

## 2. Load Raw Data & Apply the Production Feature Engineering Step

In [ ]:
df = pd.read_csv("../data/raw/customer_churn_raw.csv")
df.columns = df.columns.str.strip().str.replace(" ", "_")

df = add_features(df)
df.head()

Note that `TotalRevenue`, `AvgMobileRevenue`, and `AvgFIXRevenue` are
already gone from `df` — `add_features` drops them after deriving the
ratio/flag features below, so they aren't duplicated in the model input.
`CHURN` is still in its raw `Yes`/`No` form here since target encoding
happens later in `preprocess_data()`.

## 3. Revenue per Subscriber vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="CHURN", y="Revenue_per_Sub", data=df)
plt.title("Revenue per Subscriber vs Churn")
plt.show()

**Observation:** if churners cluster at a different revenue-per-subscriber
level than retained customers, this feature is carrying signal beyond raw
`TotalRevenue` alone — it normalizes spend by how many subscriptions
generated it, which raw revenue can't distinguish.

## 4. Engagement Score vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="CHURN", y="Engagement_Score", data=df)
plt.title("Engagement Score vs Churn")
plt.show()

**Observation:** `Engagement_Score` is the fraction of a customer's
subscriptions that are active. A visible gap between the churn and
no-churn groups here would support the EDA finding that medium-engagement
customers are more churn-prone than highly-engaged ones.

## 5. High Inactive Subscriber Flag vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x="High_Inactive_Flag", hue="CHURN", data=df)
plt.title("High Inactive Flag vs Churn")
plt.show()

**Observation:** check whether the churn rate (not just the raw count)
differs between customers with and without inactive subscriptions —
counts alone are misleading given the ~93/7 class imbalance.

## 6. Mobile Revenue Ratio vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="CHURN", y="Mobile_Revenue_Ratio", data=df)
plt.title("Mobile Revenue Ratio vs Churn")
plt.show()

**Observation:** this captures dependency on mobile services specifically,
distinct from `AvgMobileRevenue` in absolute terms (which is dropped after
this feature is derived).

## 7. FIX Service Usage Flag vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x="FIX_User", hue="CHURN", data=df)
plt.title("FIX User Flag vs Churn")
plt.show()

**Observation:** EDA suggested fixed-line users churn less. If the churn
*rate* within `FIX_User == 1` is visibly lower than within `FIX_User == 0`,
that supports keeping this as a standalone signal rather than relying only
on `AvgFIXRevenue`, which is dropped from the model after this point.

## 8. Multi-Service Flag vs Churn

`Multi_Service` is defined in `src/feature_engineering.py` as customers
with **both** mobile and FIX revenue greater than zero — a service-mix
definition, not a subscription-count definition. This is a deliberate
choice so the feature name matches what it actually measures (an earlier
draft of this feature, based on subscription counts, is not what ships in
production).

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x="Multi_Service", hue="CHURN", data=df)
plt.title("Multi-Service Flag vs Churn")
plt.show()

## 9. High-Value Customer Flag vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x="High_Value_Customer", hue="CHURN", data=df)
plt.title("High-Value Customer Flag (Top 10% Revenue) vs Churn")
plt.show()

**Observation:** EDA found high-revenue customers churn *more*, not less —
this flag exists specifically to let the model treat top-decile customers
differently, since losing them is disproportionately costly even if they
aren't more numerous.

## 10. Value × Engagement Interaction vs Churn

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x="CHURN", y="Value_Engagement_Interaction", data=df)
plt.title("Value x Engagement Interaction vs Churn")
plt.show()

**Observation:** this interaction term is designed to surface high-spend,
low-engagement customers — a profile that combines revenue importance with
elevated churn risk in a way neither raw feature captures alone.

## 11. Why ARPU / Revenue Quantile Buckets Were Not Carried Into Production

Earlier exploration considered bucketing `ARPU` and `TotalRevenue` into
Low/Medium/High quantile categories (`ARPU_Category`,
`Customer_Revenue_Segment`) on the theory that this would help a **linear**
model capture non-linear churn patterns. The production model is a Random
Forest, which already learns effective split points on continuous features
internally — manually-binned categorical versions only added extra
one-hot-encoded columns (and therefore extra dimensionality going into
SMOTE) without improving performance in testing. They were dropped from
`src/feature_engineering.py` for this reason and are intentionally absent
from this notebook's feature set as well.

## Summary

Each engineered feature in `src/feature_engineering.py` shows a visible,
business-interpretable relationship with churn, supporting its inclusion
in the model. This notebook is analysis/validation only — for the
executable pipeline, see `02_preprocessing.ipynb` and
`04_model_training.ipynb`.